# EDA > Pivot

<div class="alert alert-info">Create pivot tables, frequency tables, and crosstabs</div>

The `pivot` function creates frequency tables for a single variable or crosstabs for two variables. It supports various aggregation functions, normalization, and totals.

Note that `normalize` produces **proportions** (numbers between 0 and 1). Use `perc=True` when you want those shown as percentages with a `%` sign.

<!-- pyrsm-teaching-note: roadmap -->
## Teaching Roadmap

Use this notebook to teach how a pivot table changes the unit of analysis. Students should identify the row variable, column variable, values variable, and aggregation function. A useful check is whether each displayed number is a count, total, mean, proportion, or percentage.


In [1]:
import polars as pl
import pyrsm as rsm

## setup pyrsm for autoreload
%reload_ext autoreload
%autoreload 2
%aimport pyrsm

# Diamonds Dataset

In [2]:
diamonds = pl.read_parquet("https://github.com/radiant-ai-hub/pyrsm/raw/refs/heads/main/examples/data/data/diamonds.parquet")
diamonds

price,carat,clarity,cut,color,depth,table,x,y,z,date
i32,f64,enum,enum,enum,f64,f64,f64,f64,f64,date
580,0.32,"""VS1""","""Ideal""","""H""",61.0,56.0,4.43,4.45,2.71,2012-02-26
650,0.34,"""SI1""","""Very Good""","""G""",63.4,57.0,4.45,4.42,2.81,2012-02-26
630,0.3,"""VS2""","""Very Good""","""G""",63.1,58.0,4.27,4.23,2.68,2012-02-26
706,0.35,"""VVS2""","""Ideal""","""H""",59.2,56.0,4.6,4.65,2.74,2012-02-26
1080,0.4,"""VS2""","""Premium""","""F""",62.6,58.0,4.72,4.68,2.94,2012-02-26
3082,0.6,"""VVS1""","""Ideal""","""E""",62.5,53.7,5.35,5.43,3.38,2012-02-26
3328,0.88,"""SI1""","""Ideal""","""I""",61.7,56.0,6.14,6.18,3.8,2012-02-26
4229,0.93,"""SI1""","""Premium""","""E""",61.4,57.0,6.34,6.23,3.86,2012-02-26
1895,0.51,"""VVS2""","""Very Good""","""G""",63.4,57.0,5.09,5.06,3.22,2012-02-26


In [3]:
rsm.md("https://raw.githubusercontent.com/radiant-ai-hub/pyrsm/refs/heads/main/examples/data/data/diamonds_description.md")

## Diamond prices

Prices of 3,000 round cut diamonds

### Description

A dataset containing the prices and other attributes of a sample of 3000 diamonds. The variables are as follows:

### Variables

- price = price in US dollars ($338--$18,791)
- carat = weight of the diamond (0.2--3.00)
- clarity = a measurement of how clear the diamond is (I1 (worst), SI2, SI1, VS2, VS1, VVS2, VVS1, IF (best))
- cut = quality of the cut (Fair, Good, Very Good, Premium, Ideal)
- color = diamond color, from J (worst) to D (best)
- depth = total depth percentage = z / mean(x, y) = 2 * z / (x + y) (54.2--70.80)
- table = width of top of diamond relative to widest point (50--69)
- x = length in mm (3.73--9.42)
- y = width in mm (3.71--9.29)
- z = depth in mm (2.33--5.58)
- date = shipment date

### Additional information

<a href="http://www.diamondse.info/diamonds-clarity.asp" target="_blank">Diamond search engine</a>


## Frequency Table (Single Variable)

Count occurrences of each value in a categorical column.

In [4]:
rsm.eda.pivot(diamonds, rows="cut")

cut,count
enum,u32
"""Premium""",771
"""Good""",275
"""Very Good""",677
"""Fair""",101
"""Ideal""",1176


In [5]:
rsm.eda.pivot(diamonds, rows="color")

color,count
enum,u32
"""I""",284
"""H""",454
"""F""",565
"""E""",554
"""G""",597
"""J""",164
"""D""",382


In [6]:
rsm.eda.pivot(diamonds, rows="cut", values="price")

cut,price_mean
enum,f64
"""Fair""",4505.237624
"""Ideal""",3470.223639
"""Good""",4130.432727
"""Premium""",4369.40856
"""Very Good""",3959.915805


## Frequency Table with Proportions

`normalize` adds a column with each group's share of the total.

In [7]:
rsm.eda.pivot(diamonds, rows="cut", normalize="total")

cut,count,count_prop
enum,u32,f64
"""Fair""",101,0.033667
"""Premium""",771,0.257
"""Ideal""",1176,0.392
"""Good""",275,0.091667
"""Very Good""",677,0.225667


Set `perc=True` to show that share as a percentage instead.

In [8]:
rsm.eda.pivot(diamonds, rows="cut", normalize="total", perc=True)

cut,count,count_perc
enum,u32,str
"""Premium""",771,"""25.70%"""
"""Good""",275,"""9.17%"""
"""Ideal""",1176,"""39.20%"""
"""Very Good""",677,"""22.57%"""
"""Fair""",101,"""3.37%"""


## Frequency Table with Totals

In [9]:
rsm.eda.pivot(diamonds, rows="cut", totals=True)

cut,count
str,f64
"""Good""",275.0
"""Ideal""",1176.0
"""Very Good""",677.0
"""Premium""",771.0
"""Fair""",101.0
"""Total""",3000.0


## Crosstab (Two Variables)

Cross-tabulate two categorical variables to see their joint distribution.

In [10]:
rsm.eda.pivot(diamonds, rows="cut", cols="color")

cut,D,F,J,G,E,H,I
enum,f64,f64,f64,f64,f64,f64,f64
"""Very Good""",77.0,136.0,39.0,131.0,140.0,95.0,59.0
"""Fair""",15.0,17.0,7.0,16.0,14.0,21.0,11.0
"""Good""",35.0,55.0,20.0,40.0,62.0,37.0,26.0
"""Premium""",92.0,119.0,45.0,156.0,144.0,132.0,83.0
"""Ideal""",163.0,238.0,53.0,254.0,194.0,169.0,105.0


## Crosstab with Totals

In [11]:
rsm.eda.pivot(diamonds, rows="cut", cols="color", totals=True)

cut,E,H,J,G,D,I,F,Total
str,f64,f64,f64,f64,f64,f64,f64,f64
"""Ideal""",194.0,169.0,53.0,254.0,163.0,105.0,238.0,1176.0
"""Very Good""",140.0,95.0,39.0,131.0,77.0,59.0,136.0,677.0
"""Premium""",144.0,132.0,45.0,156.0,92.0,83.0,119.0,771.0
"""Fair""",14.0,21.0,7.0,16.0,15.0,11.0,17.0,101.0
"""Good""",62.0,37.0,20.0,40.0,35.0,26.0,55.0,275.0
"""Total""",554.0,454.0,164.0,597.0,382.0,284.0,565.0,3000.0


## Row Normalization

Show proportions within each row (rows sum to 1).

In [12]:
rsm.eda.pivot(diamonds, rows="cut", cols="color", normalize="row", totals=True)

cut,H,E,I,F,G,J,D,Total
str,f64,f64,f64,f64,f64,f64,f64,f64
"""Ideal""",0.143707,0.164966,0.089286,0.202381,0.215986,0.045068,0.138605,1.0
"""Good""",0.134545,0.225455,0.094545,0.2,0.145455,0.072727,0.127273,1.0
"""Fair""",0.207921,0.138614,0.108911,0.168317,0.158416,0.069307,0.148515,1.0
"""Premium""",0.171206,0.18677,0.107652,0.154345,0.202335,0.058366,0.119326,1.0
"""Very Good""",0.140325,0.206795,0.087149,0.200886,0.193501,0.057607,0.113737,1.0
"""Total""",0.151333,0.184667,0.094667,0.188333,0.199,0.054667,0.127333,1.0


## Column Normalization

Show proportions within each column (columns sum to 1).

In [13]:
rsm.eda.pivot(diamonds, rows="cut", cols="color", normalize="column")

cut,E,I,G,H,J,D,F
enum,f64,f64,f64,f64,f64,f64,f64
"""Very Good""",0.252708,0.207746,0.21943,0.209251,0.237805,0.201571,0.240708
"""Fair""",0.025271,0.038732,0.026801,0.046256,0.042683,0.039267,0.030088
"""Ideal""",0.350181,0.369718,0.425461,0.372247,0.323171,0.426702,0.421239
"""Premium""",0.259928,0.292254,0.261307,0.290749,0.27439,0.240838,0.210619
"""Good""",0.111913,0.091549,0.067002,0.081498,0.121951,0.091623,0.097345


## Showing Percentages

`perc=True` formats the cells as percentages, and `dec` sets the number of decimals. This is display-only: without it you get the raw proportions shown above.

In [14]:
rsm.eda.pivot(diamonds, rows="cut", cols="color", normalize="row", totals=True, perc=True)

cut,J,F,E,H,G,I,D,Total
str,str,str,str,str,str,str,str,str
"""Good""","""7.27%""","""20.00%""","""22.55%""","""13.45%""","""14.55%""","""9.45%""","""12.73%""","""100.00%"""
"""Premium""","""5.84%""","""15.43%""","""18.68%""","""17.12%""","""20.23%""","""10.77%""","""11.93%""","""100.00%"""
"""Very Good""","""5.76%""","""20.09%""","""20.68%""","""14.03%""","""19.35%""","""8.71%""","""11.37%""","""100.00%"""
"""Fair""","""6.93%""","""16.83%""","""13.86%""","""20.79%""","""15.84%""","""10.89%""","""14.85%""","""100.00%"""
"""Ideal""","""4.51%""","""20.24%""","""16.50%""","""14.37%""","""21.60%""","""8.93%""","""13.86%""","""100.00%"""
"""Total""","""5.47%""","""18.83%""","""18.47%""","""15.13%""","""19.90%""","""9.47%""","""12.73%""","""100.00%"""


In [15]:
rsm.eda.pivot(
    diamonds, rows="cut", cols="color", normalize="column", perc=True, dec=1
)

cut,H,J,I,E,F,D,G
enum,str,str,str,str,str,str,str
"""Good""","""8.1%""","""12.2%""","""9.2%""","""11.2%""","""9.7%""","""9.2%""","""6.7%"""
"""Fair""","""4.6%""","""4.3%""","""3.9%""","""2.5%""","""3.0%""","""3.9%""","""2.7%"""
"""Very Good""","""20.9%""","""23.8%""","""20.8%""","""25.3%""","""24.1%""","""20.2%""","""21.9%"""
"""Ideal""","""37.2%""","""32.3%""","""37.0%""","""35.0%""","""42.1%""","""42.7%""","""42.5%"""
"""Premium""","""29.1%""","""27.4%""","""29.2%""","""26.0%""","""21.1%""","""24.1%""","""26.1%"""


## Aggregation with Values

Instead of counting, aggregate a numeric variable by groups.

In [16]:
# Mean price by cut and color
rsm.eda.pivot(diamonds, rows="cut", cols="color", values="price", agg="mean")

cut,D,H,F,G,J,I,E
enum,f64,f64,f64,f64,f64,f64,f64
"""Good""",3436.514286,3958.162162,3443.4,5116.225,3837.25,6147.346154,3847.209677
"""Ideal""",2667.478528,3515.674556,3375.054622,3844.535433,4987.754717,4330.352381,2851.659794
"""Fair""",4582.733333,5742.47619,5101.294118,3919.8125,6102.0,2676.727273,3149.928571
"""Premium""",3814.98913,5066.295455,4086.831933,3976.5,7515.466667,5056.686747,3364.694444
"""Very Good""",3299.974026,4207.294737,3669.727941,3864.274809,5212.410256,5409.881356,3566.442857


In [17]:
# Median carat by cut and color
rsm.eda.pivot(diamonds, rows="cut", cols="color", values="carat", agg="median")

cut,H,F,I,J,D,E,G
enum,f64,f64,f64,f64,f64,f64,f64
"""Good""",1.0,0.7,1.36,0.865,0.7,0.7,1.0
"""Ideal""",0.7,0.54,0.7,1.07,0.51,0.51,0.535
"""Premium""",1.02,0.71,1.01,1.51,0.69,0.52,0.71
"""Very Good""",0.9,0.7,1.01,1.04,0.54,0.71,0.72
"""Fair""",1.01,1.0,0.73,1.0,0.9,0.715,0.855


# Titanic Dataset

In [18]:
titanic = pl.read_parquet("https://github.com/radiant-ai-hub/pyrsm/raw/refs/heads/main/examples/data/data/titanic.parquet")
titanic.head()

pclass,survived,sex,age,sibsp,parch,fare,name,cabin,embarked
enum,enum,enum,f64,i32,i32,f64,str,str,enum
"""1st""","""Yes""","""female""",29.0,0,0,211.337494,"""Allen, Miss. Elisabeth Walton""","""B5""","""Southampton"""
"""1st""","""Yes""","""male""",0.9167,1,2,151.550003,"""Allison, Master. Hudson Trevor""","""C22 C26""","""Southampton"""
"""1st""","""No""","""female""",2.0,1,2,151.550003,"""Allison, Miss. Helen Loraine""","""C22 C26""","""Southampton"""
"""1st""","""No""","""male""",30.0,1,2,151.550003,"""Allison, Mr. Hudson Joshua Crei""","""C22 C26""","""Southampton"""
"""1st""","""No""","""female""",25.0,1,2,151.550003,"""Allison, Mrs. Hudson J C (Bessi""","""C22 C26""","""Southampton"""


In [19]:
rsm.md("https://raw.githubusercontent.com/radiant-ai-hub/pyrsm/refs/heads/main/examples/data/data/titanic_description.md")

## Titanic

This dataset describes the survival status of individual passengers on the Titanic. The titanic data frame does not contain information from the crew, but it does contain actual ages of (some of) the passengers. The principal source for data about Titanic passengers is the Encyclopedia Titanica. One of the original sources is Eaton & Haas (1994) Titanic: Triumph and Tragedy, Patrick Stephens Ltd, which includes a passenger list created by many researchers and edited by Michael A. Findlay.

## Variables

* survival - Survival (Yes, No)
* pclass - Passenger Class (1st, 2nd, 3rd)
* sex - Sex (female, male)
* age - Age in years
* sibsp - Number of Siblings/Spouses Aboard
* parch - Number of Parents/Children Aboard
* fare - Passenger Fare
* name - Name
* cabin - Cabin
* embarked - Port of Embarkation (Cherbourg, Queenstown, Southampton)

##  Notes

`pclass` is a proxy for socio-economic status (SES) 1st ~ Upper; 2nd ~ Middle; 3rd ~ Lower

Age is in Years; Fractional if Age less than One (1). If the Age is Estimated, it is in the form xx.5

With respect to the family relation variables (i.e. sibsp and parch) some relations were ignored.  The following are the definitions used for sibsp and parch.

Sibling:  Brother, Sister, Stepbrother, or Stepsister of Passenger Aboard Titanic
Spouse:   Husband or Wife of Passenger Aboard Titanic (Mistresses and Fiances Ignored)
Parent:   Mother or Father of Passenger Aboard Titanic
Child:    Son, Daughter, Stepson, or Stepdaughter of Passenger Aboard Titanic

Other family relatives excluded from this study include cousins, nephews/nieces, aunts/uncles, and in-laws. Some children travelled only with a nanny, therefore parch=0 for them.  As well, some travelled with very close friends or neighbors in a village, however, the definitions do not support such relations.

Note: Missing values and the `ticket` variable were removed from the data

## Related reading

<a href="http://phys.org/news/2012-07-shipwrecks-men-survive.html" target="_blank">In shipwrecks, men more likely to survive</a>

## Survival by Passenger Class

In [20]:
rsm.eda.pivot(titanic, rows="pclass", cols="survived", totals=True)

pclass,Yes,No,Total
str,f64,f64,f64
"""2nd""",115.0,146.0,261.0
"""3rd""",131.0,369.0,500.0
"""1st""",179.0,103.0,282.0
"""Total""",425.0,618.0,1043.0


## Survival Rate by Class (Row Proportions)

In [21]:
rsm.eda.pivot(titanic, rows="pclass", cols="survived", normalize="row", totals=True)

pclass,No,Yes,Total
str,f64,f64,f64
"""2nd""",0.559387,0.440613,1.0
"""1st""",0.365248,0.634752,1.0
"""3rd""",0.738,0.262,1.0
"""Total""",0.592522,0.407478,1.0


The same table as percentages:

In [22]:
rsm.eda.pivot(
    titanic, rows="pclass", cols="survived", normalize="row", totals=True, perc=True
)

pclass,Yes,No,Total
str,str,str,str
"""2nd""","""44.06%""","""55.94%""","""100.00%"""
"""1st""","""63.48%""","""36.52%""","""100.00%"""
"""3rd""","""26.20%""","""73.80%""","""100.00%"""
"""Total""","""40.75%""","""59.25%""","""100.00%"""


## Survival by Sex

In [23]:
rsm.eda.pivot(titanic, rows="sex", cols="survived", normalize="row", totals=True)

sex,Yes,No,Total
str,f64,f64,f64
"""male""",0.205479,0.794521,1.0
"""female""",0.751295,0.248705,1.0
"""Total""",0.407478,0.592522,1.0


## Embarkation Port Distribution

In [24]:
rsm.eda.pivot(titanic, rows="embarked", normalize="total", totals=True)

embarked,count,count_prop
str,f64,f64
"""Cherbourg""",212.0,0.20326
"""Queenstown""",50.0,0.047939
"""Southampton""",781.0,0.748802
"""Total""",1043.0,1.0


## Mean Fare by Class and Survival

In [25]:
rsm.eda.pivot(titanic, rows="pclass", cols="survived", values="fare", agg="mean")

pclass,No,Yes
enum,f64,f64
"""3rd""",13.039712,12.427449
"""1st""",74.678276,102.465226
"""2nd""",20.811044,23.180471


© Vincent Nijs (2026)

## Additional examples


In [26]:
# Normalized crosstab with totals
rsm.eda.pivot(diamonds, rows="cut", cols="color", normalize="row", totals=True)


cut,F,D,G,H,E,J,I,Total
str,f64,f64,f64,f64,f64,f64,f64,f64
"""Premium""",0.154345,0.119326,0.202335,0.171206,0.18677,0.058366,0.107652,1.0
"""Very Good""",0.200886,0.113737,0.193501,0.140325,0.206795,0.057607,0.087149,1.0
"""Ideal""",0.202381,0.138605,0.215986,0.143707,0.164966,0.045068,0.089286,1.0
"""Good""",0.2,0.127273,0.145455,0.134545,0.225455,0.072727,0.094545,1.0
"""Fair""",0.168317,0.148515,0.158416,0.207921,0.138614,0.069307,0.108911,1.0
"""Total""",0.188333,0.127333,0.199,0.151333,0.184667,0.054667,0.094667,1.0
